# 10장 · 🎯 전체 조립 — 미니 트랜스포머 🏁

웹 챕터 **"전체 조립"** 과 짝을 이루는 **마지막** 실습입니다.
지금까지 배운 조각(임베딩·위치·어텐션·블록·출력층)을 모두 이어, **작동하는 미니 트랜스포머**를
numpy 로 처음부터 끝까지 만들어 봅니다. 이 학습의 도착점이에요!

> ⚠️ 이 모델은 **학습을 안 한**(가중치가 무작위) 상태라 생성 단어는 의미가 없습니다.
> 여기서 볼 것은 **"구조가 입력부터 출력까지 끝까지 돈다"** 는 것. 학습은 다음 챕터(ch11)에서!

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요
from tutor import chat

chat('ch10')

## 1. 사전과 가중치 준비

작은 사전 8단어, 임베딩 차원 16. 모든 가중치는 재현을 위해 시드를 고정해 무작위로 만듭니다.

In [1]:
import numpy as np
rng = np.random.default_rng(42)

VOCAB = ['<s>', '나는', '너를', '강아지', '공을', '좋다', '물었다', '.']
V, d = len(VOCAB), 16
stoi = {w: i for i, w in enumerate(VOCAB)}

E  = rng.normal(0, 0.5, (V, d))          # 토큰 임베딩 (출력층과 공유 = weight tying)
Wq = rng.normal(0, 0.3, (d, d))
Wk = rng.normal(0, 0.3, (d, d))
Wv = rng.normal(0, 0.3, (d, d))
W1 = rng.normal(0, 0.3, (d, 4*d)); b1 = np.zeros(4*d)   # FFN: d -> 4d
W2 = rng.normal(0, 0.3, (4*d, d)); b2 = np.zeros(d)     # FFN: 4d -> d
print('사전', V, '단어, 임베딩 차원', d)

사전 8 단어, 임베딩 차원 16


## 2. 부품 함수들 (배운 그대로)

- **위치 인코딩**(ch4): 사인·코사인
- **셀프 어텐션**(ch6): Q·K·V + 인과 마스킹(뒤 단어는 못 보게)
- **층정규화·FFN**(ch8)

In [2]:
def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z); return e / e.sum(axis=-1, keepdims=True)

def positional_encoding(n, d):
    pe = np.zeros((n, d))
    for pos in range(n):
        for i in range(d):
            ang = pos / (10000 ** (2*(i//2)/d))
            pe[pos, i] = np.sin(ang) if i % 2 == 0 else np.cos(ang)
    return pe

def layer_norm(x, eps=1e-5):
    mu = x.mean(-1, keepdims=True); sd = x.std(-1, keepdims=True)
    return (x - mu) / (sd + eps)

def self_attention(X):
    Q, K, Vv = X @ Wq, X @ Wk, X @ Wv
    scores = Q @ K.T / np.sqrt(d)
    mask = np.triu(np.full_like(scores, -1e9), k=1)   # 인과 마스킹
    W = softmax(scores + mask)
    return W @ Vv

def ffn(X):
    return np.maximum(X @ W1 + b1, 0) @ W2 + b2        # ReLU

def block(X):
    X = layer_norm(X + self_attention(X))              # Add & Norm
    X = layer_norm(X + ffn(X))                         # Add & Norm
    return X

## 3. 순전파: 토큰들 → 다음 단어 점수

임베딩 + 위치 → 블록 → (마지막 자리 벡터) · 임베딩ᵀ = 다음 단어 logits (weight tying).

In [3]:
def forward(ids):
    X = E[ids] + positional_encoding(len(ids), d)   # 임베딩 + 위치
    X = block(X)                                    # 트랜스포머 블록 (×1)
    logits = X[-1] @ E.T                            # 마지막 자리 → 출력층(공유 임베딩)
    return logits

ids = [stoi['<s>'], stoi['강아지'], stoi['공을']]
print('입력:', [VOCAB[i] for i in ids])
logits = forward(ids)
probs = softmax(logits)
for w, p in sorted(zip(VOCAB, probs), key=lambda x: -x[1])[:4]:
    print(f'  {w:5s}: {p*100:4.1f}%')
print('다음 단어(무작위 가중치):', VOCAB[int(np.argmax(probs))])

입력: ['<s>', '강아지', '공을']
  물었다  : 36.6%
  강아지  : 30.1%
  공을   : 21.5%
  너를   :  9.2%
다음 단어(무작위 가중치): 물었다


## 4. 자기회귀 생성 (ch1의 그 반복!)

다음 단어를 고르고 → 붙이고 → 다시 순전파. 마침표나 최대 길이까지 반복합니다.
(학습 전이라 의미는 없지만, **입력→출력→반복** 구조가 완결됩니다)

In [4]:
def generate(start='강아지', max_len=6):
    ids = [stoi['<s>'], stoi[start]]
    for _ in range(max_len):
        nxt = int(np.argmax(softmax(forward(ids))))
        ids.append(nxt)
        if VOCAB[nxt] == '.':
            break
    return [VOCAB[i] for i in ids]

print('생성:', ' '.join(generate('강아지')))

생성: <s> 강아지 강아지 물었다 물었다 물었다 물었다 물었다


## 5. 조립 점검 ✏️

우리 미니 트랜스포머가 **끝까지 도는지** 확인합니다: 임베딩→위치→어텐션→블록→출력층.

In [5]:
ids = [stoi['<s>'], stoi['나는'], stoi['너를']]
X0 = E[ids] + positional_encoding(len(ids), d)
X1 = block(X0)
logits = X1[-1] @ E.T
probs = softmax(logits)

print('임베딩+위치 shape:', X0.shape, '(단어수 x 차원)')
print('블록 출력 shape  :', X1.shape)
print('logits 개수      :', logits.shape[0], '= 사전 크기', V)
assert logits.shape[0] == V, '단어마다 점수 하나!'
assert abs(probs.sum() - 1) < 1e-9, '확률 합은 1!'
print('✅ 입력 → 임베딩 → 위치 → 어텐션 → 블록 → 출력층 → 확률, 끝까지 돌았습니다!')

임베딩+위치 shape: (3, 16) (단어수 x 차원)
블록 출력 shape  : (3, 16)
logits 개수      : 8 = 사전 크기 8
✅ 입력 → 임베딩 → 위치 → 어텐션 → 블록 → 출력층 → 확률, 끝까지 돌았습니다!


---
## 🎓 도착점

**토큰 하나가 벡터가 되고 → 서로 참고하고 → 블록을 지나 → 다음 단어 확률이 되기까지**,
LLM의 속을 처음부터 끝까지 손으로 따라왔습니다.

지금까지 지나온 길: 토큰화(ch2) → 임베딩(ch3) → 위치(ch4) → 어텐션(ch5·6) → 멀티헤드(ch7)
→ 블록(ch8) → 출력층(ch9) → **전체 조립(ch10)**. 🏁

더 나아가려면: **ch11 학습**(무작위 가중치가 어떻게 똑똑해지나) · **ch12 샘플링**(다음 단어 고르는 법).
이제 ChatGPT를 보면 말할 수 있어요. *"아, 저건 다음 단어를 확률로 고르는 거지."* 🙂